In [8]:
%pip install streamlit==1.50.0 pandas==2.3.3 numpy==2.0.2 scikit-learn==1.6.1 joblib==1.5.3 ipywidgets==8.1.9

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import io
import base64
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef, confusion_matrix, classification_report)

# 1. Load built-in Breast Cancer Dataset
dataset = load_breast_cancer(as_frame=True)
FEATURES = list(dataset.feature_names)
LABELS = {0: "malignant", 1: "benign"}

# 2. Split dataset into Train (80%) and Test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    dataset.data, dataset.target, test_size=0.2, random_state=42,
    stratify=dataset.target
)

# 3. Train all 5 Classifiers
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=42)),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "Gaussian Naive Bayes": GaussianNB(),
    "Support Vector Machine": make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
}

for name, model in models.items():
    model.fit(X_train, y_train)

# Prepare built-in test data
builtin_test = X_test.copy()
builtin_test["target"] = y_test
current_test = builtin_test.copy()

# UI Widgets
chooser = widgets.Dropdown(options=list(models.keys()), description="Classifier:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="400px"))
output = widgets.Output()

def render(change=None):
    with output:
        clear_output(wait=True)
        try:
            df = current_test.copy()
            X = df[FEATURES].apply(pd.to_numeric)
            y = df["target"].astype(int)
            
            model = models[chooser.value]
            pred = model.predict(X)
            prob = model.predict_proba(X)[:, list(model.classes_).index(1)]
            
            metrics = {
                "Accuracy": accuracy_score(y, pred),
                "AUC": roc_auc_score(y, prob) if y.nunique() == 2 else np.nan,
                "Precision": precision_score(y, pred, zero_division=0),
                "Recall": recall_score(y, pred, zero_division=0),
                "F1 Score": f1_score(y, pred, zero_division=0),
                "MCC": matthews_corrcoef(y, pred),
            }
            
            display(Markdown(f"## Selected: **{chooser.value}**"))
            display(Markdown(f"**Test Sample Size:** {len(df)} rows | Positive Class: benign (1)"))
            display(pd.DataFrame([metrics]).round(4))
            
            display(Markdown("### Confusion Matrix"))
            display(pd.DataFrame(confusion_matrix(y, pred, labels=[0, 1]),
                index=["Actual malignant", "Actual benign"],
                columns=["Predicted malignant", "Predicted benign"]))
            
            display(Markdown("### Classification Report"))
            print(classification_report(y, pred, labels=[0, 1],
                target_names=["malignant", "benign"], zero_division=0))
            
            result = df.copy()
            result["predicted_target"] = pred
            result["predicted_label"] = [LABELS[int(v)] for v in pred]
            
            display(Markdown("### Sample Predictions (First 10 rows)"))
            columns = ["target", "predicted_target", "predicted_label"] + FEATURES[:4]
            display(result[columns].head(10))
            
        except Exception as error:
            display(Markdown(f"**Error:** {error}"))

chooser.observe(render, names="value")

# Display Title and Dashboard
display(Markdown("# 🔬 Machine Learning Classification Lab"))
display(Markdown("Five trained models on Wisconsin Diagnostic Breast Cancer dataset."))
display(chooser, output)
render()

# 🔬 Machine Learning Classification Lab

Five trained models on Wisconsin Diagnostic Breast Cancer dataset.

Dropdown(description='Classifier:', layout=Layout(width='400px'), options=('Logistic Regression', 'Decision Tr…

Output()